### Connect to Ollama

In [35]:
import ollama

### Connect to Qdrant


In [32]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

### Add new collection

In [ ]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="ollama_test_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

### Process pdf file


In [ ]:
import pypdf
import os




PDF 1 length: 8318
PDF 1 preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior trou
PDF 2 length: 7140
PDF 2 preview: INFORMATION TECHNOLOGY MANAGER
Professional Summary
Possesses an extensive background in Information
PDF 3 length: 5192
PDF 3 preview: WORKING RF SYSTEMS ENGINEER
Qualifications
Microsoft office/Office for Mac, pages, numbers, keynote 
PDF 4 length: 5115
PDF 4 preview: INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated 
IT Manager
 
well-versed in analyzing and mitigati
PDF 5 length: 5377
PDF 5 preview: IT MANAGEMENT
Career Overview
Detail-oriented professional with extensive Information Technology exp


### Create a collection for each CV

In [ ]:
for idx, doc_text in enumerate(all_texts):
    client.create_collection(
        collection_name=f"ollama_test_collection_{idx}",
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )

### Divide all texts into chunks of 512 characters

In [29]:
for idx, doc_text in enumerate(all_texts):
    chunks = [doc_text[i:i+512] for i in range(0, len(doc_text), 512)]
    for chunk_idx, chunk in enumerate(chunks):
        embedding = ollama.embeddings(
            model='mxbai-embed-large',
            prompt=chunk,
        )['embedding']

        client.upsert(
            collection_name=f"ollama_test_collection_{idx}",
            wait=True,
            points=[{
                "id": idx * 100000 + chunk_idx,
                "vector": embedding,
                "payload": {"text": chunk},
            }],
        )

### Compare CV-level scoring approaches for each job posting

In [41]:
import math

job_postings_path = '../data/job_postings'

job_postings = {}
for filename in sorted(os.listdir(job_postings_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(job_postings_path, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            job_postings[filename] = file.read()


def mean_all(scores):
    return sum(scores) / len(scores)


def topk_mean(scores, k=5):
    k = min(k, len(scores))
    return sum(scores[:k]) / k


def weighted_topk_mean(scores, k=5):
    k = min(k, len(scores))
    top_scores = scores[:k]
    weights = list(range(k, 0, -1))
    return sum(score * weight for score, weight in zip(top_scores, weights)) / sum(weights)


def softmax_pooling(scores, alpha=10.0):
    max_score = max(scores)
    stabilized = [math.exp(alpha * (score - max_score)) for score in scores]
    return max_score + math.log(sum(stabilized)) / alpha


def hybrid_max_topk(scores, k=5, w=0.4):
    return w * max(scores) + (1 - w) * topk_mean(scores, k=k)


scoring_methods = {
    'max_score': lambda scores: max(scores),
    'mean_all': lambda scores: mean_all(scores),
    'topk_mean_k5': lambda scores: topk_mean(scores, k=5),
    'weighted_topk_mean_k5': lambda scores: weighted_topk_mean(scores, k=5),
    'softmax_pooling_alpha10': lambda scores: softmax_pooling(scores, alpha=10.0),
    'hybrid_max_topk_k5_w0.4': lambda scores: hybrid_max_topk(scores, k=5, w=0.4),
}

for posting_name, posting_text in job_postings.items():
    posting_embedding = ollama.embeddings(
        model='mxbai-embed-large',
        prompt=posting_text,
    )['embedding']

    cv_results = []

    for idx in range(len(all_texts)):
        collection_name = f"ollama_test_collection_{idx}"

        point_count = client.count(
            collection_name=collection_name,
            exact=True,
        ).count

        if point_count == 0:
            continue

        points = client.query_points(
            collection_name=collection_name,
            query=posting_embedding,
            with_payload=True,
            limit=point_count,
        ).points

        if not points:
            continue

        scores = [point.score for point in points]
        top_point = points[0]

        method_scores = {
            method_name: method_fn(scores)
            for method_name, method_fn in scoring_methods.items()
        }

        cv_results.append({
            'collection_name': collection_name,
            'cv_index': idx,
            'point_count': point_count,
            'top_chunk_text': top_point.payload.get('text', '') if top_point.payload else '',
            'method_scores': method_scores,
        })

    print(f"Job posting: {posting_name}")

    if not cv_results:
        print("  No matches found.")
        print('-' * 100)
        continue

    for method_name in scoring_methods.keys():
        best_cv = max(cv_results, key=lambda row: row['method_scores'][method_name])
        best_score = best_cv['method_scores'][method_name]

        print(f"  Method: {method_name}")
        print(f"    Best CV: {best_cv['collection_name']} (CV #{best_cv['cv_index'] + 1})")
        print(f"    Score: {best_score:.4f}")
        print(f"    Points in CV collection: {best_cv['point_count']}")
        print(f"    Top chunk preview: {best_cv['top_chunk_text'][:180]}")

    print('-' * 100)

Job posting: cloud_systems_administrator.txt
  Method: max_score
    Best CV: ollama_test_collection_0 (CV #1)
    Score: 0.6747
    Points in CV collection: 17
    Top chunk preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network securit
  Method: mean_all
    Best CV: ollama_test_collection_0 (CV #1)
    Score: 0.6106
    Points in CV collection: 17
    Top chunk preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network securit
  Method: topk_mean_k5
    Best CV: ollama_test_collection_0 (CV #1)
    Score: 0.6593
    Points in CV collection: 17
    Top chunk preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network s